# Day 3 — PySpark: Date & Time Functions
**Topics:** to_date, datediff, date_trunc, date_format, year/month, date_add, current_date, months_between

In [ ]:
import os
os.environ['JAVA_HOME']           = 'C:/Program Files/DBeaver/jre'
os.environ['PYSPARK_PYTHON']        = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\hariom\AppData\Local\Programs\Python\Python311\python.exe'

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit,
    to_date, datediff, date_trunc, date_format,
    year, month, dayofmonth, dayofweek,
    date_add, date_sub,
    months_between, add_months,
    current_date,
    min as spark_min, max as spark_max,
    sum as spark_sum, count,
)
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, DateType

spark = SparkSession.builder \
    .master('local[*]') \
    .appName('Day3_PySpark_Dates') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

## Sample Data — Orders

In [ ]:
data = [
    (1, 1001, '2026-05-01', '2026-05-05', 250.00),
    (2, 1002, '2026-05-03', '2026-05-14', 189.50),   # slow: 11 days
    (3, 1001, '2026-05-10', '2026-05-17', 310.00),   # slow: 7 days
    (4, 1003, '2026-05-15', '2026-05-23', 95.00),    # slow: 8 days
    (5, 1004, '2026-05-20', '2026-05-21', 540.00),
    (6, 1002, '2026-06-01', '2026-06-03', 220.00),
    (7, 1005, '2026-06-05', '2026-06-19', 75.00),    # slow: 14 days
    (8, 1003, '2026-06-10', '2026-06-12', 430.00),
    (9, 1006, '2026-06-12', '2026-06-14', 180.00),
    (10,1007, '2026-06-15', '2026-06-16', 99.00),
    (11,1001, '2025-01-05', '2025-01-07', 500.00),
    (12,1002, '2025-03-12', '2025-03-14', 320.00),
    (13,1003, '2025-06-20', '2025-07-01', 150.00),
    (14,1004, '2025-09-08', '2025-09-10', 780.00),
    (15,1005, '2025-11-25', '2025-11-30', 210.00),
    (16,1001, '2025-12-01', '2025-12-15', 640.00),   # slow: 14 days
    (17,1002, '2025-12-20', '2025-12-22', 110.00),
    (18,1004, '2026-01-10', '2026-01-12', 920.00),
    (19,1001, '2026-02-14', '2026-02-16', 275.00),
    (20,1003, '2026-03-01', '2026-03-04', 360.00),
    (21,1002, '2026-03-18', '2026-03-20', 490.00),
    (22,1005, '2026-04-05', '2026-04-07', 130.00),
]

schema = StructType([
    StructField('order_id',    IntegerType(), False),
    StructField('customer_id', IntegerType(), False),
    StructField('order_date',  StringType(),  False),
    StructField('ship_date',   StringType(),  True),
    StructField('amount',      DoubleType(),  False),
])

df_raw = spark.createDataFrame(data, schema)

df_orders = df_raw \
    .withColumn('order_date', to_date(col('order_date'), 'yyyy-MM-dd')) \
    .withColumn('ship_date',  to_date(col('ship_date'),  'yyyy-MM-dd'))

df_orders.printSchema()
df_orders.show(5)

## 1. datediff — Days between two dates

In [ ]:
# datediff(end_col, start_col) — end FIRST!
df_diff = df_orders.withColumn(
    'delivery_days', datediff(col('ship_date'), col('order_date'))
)

df_diff.select('order_id', 'order_date', 'ship_date', 'delivery_days') \
       .orderBy('delivery_days', ascending=False) \
       .show(8)

## 2. date_add / date_sub

In [ ]:
df_dates = df_orders.withColumn('expected_ship_7d',  date_add(col('order_date'), 7)) \
                    .withColumn('expected_ship_30d', date_add(col('order_date'), 30)) \
                    .withColumn('ninety_days_ago',   date_sub(current_date(), 90))

df_dates.select('order_date', 'expected_ship_7d', 'expected_ship_30d', 'ninety_days_ago').show(4)

## 3. current_date — filter recent orders

In [ ]:
# Orders placed in the last 90 days
df_recent = df_orders.filter(col('order_date') >= date_sub(current_date(), 90))
print('Recent orders count:', df_recent.count())
df_recent.orderBy('order_date', ascending=False).show()

## 4. date_trunc — truncate to month/year

In [ ]:
df_trunc = df_orders \
    .withColumn('month_start',   date_trunc('month',   col('order_date'))) \
    .withColumn('year_start',    date_trunc('year',    col('order_date'))) \
    .withColumn('quarter_start', date_trunc('quarter', col('order_date')))

df_trunc.select('order_date', 'month_start', 'year_start', 'quarter_start').show(5)

## 5. date_format — format as string (Java codes)

In [ ]:
df_fmt = df_orders \
    .withColumn('ym',        date_format(col('order_date'), 'yyyy-MM')) \
    .withColumn('label',     date_format(col('order_date'), 'MMM yyyy')) \
    .withColumn('full_date', date_format(col('order_date'), 'dd MMMM yyyy'))

df_fmt.select('order_date', 'ym', 'label', 'full_date').show(5)

## 6. year / month / dayofmonth — extract fields

In [ ]:
df_extract = df_orders \
    .withColumn('yr',  year(col('order_date'))) \
    .withColumn('mo',  month(col('order_date'))) \
    .withColumn('dy',  dayofmonth(col('order_date'))) \
    .withColumn('dow', dayofweek(col('order_date')))   # 1=Sun, 2=Mon ...

df_extract.select('order_date', 'yr', 'mo', 'dy', 'dow').show(5)

## 7. Monthly revenue aggregation

In [ ]:
df_monthly = (
    df_orders
    .withColumn('month_ts', date_trunc('month', col('order_date')))
    .withColumn('month',    date_format(col('order_date'), 'yyyy-MM'))
    .groupBy('month_ts', 'month')
    .agg(
        spark_sum('amount').alias('total_revenue'),
        count('*').alias('num_orders'),
    )
    .orderBy('month_ts')
    .select('month', 'total_revenue', 'num_orders')
)

df_monthly.show()

## 8. First order per customer — spark_min on dates

In [ ]:
df_first = (
    df_orders
    .groupBy('customer_id')
    .agg(
        spark_min('order_date').alias('first_order_date'),
        spark_max('order_date').alias('last_order_date'),
        count('*').alias('total_orders'),
        spark_sum('amount').alias('lifetime_value'),
    )
    .orderBy('lifetime_value', ascending=False)
)

df_first.show()

## 9. months_between — tenure in months

In [ ]:
df_tenure = (
    df_first
    .withColumn('tenure_months',
                months_between(col('last_order_date'), col('first_order_date')))
)

df_tenure.select('customer_id', 'first_order_date', 'last_order_date', 'tenure_months').show()